In [ ]:
import os
os.environ["WANDB_START_METHOD"] = "thread"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

In [ ]:
WANDB_ENABLED = False  # set True to log runs to WandB

RUN_MODELS = {
    'logistic_regression': True,
    'decision_tree':       True,
    'random_forest':       True,
    'gradient_boosting':   True,
}

In [ ]:
df = pd.read_csv(r"C:\Users\17063\OneDrive\Documents\GitHub\DS_Capstone_Group_1\data\v2_cleaned\all_merged.csv")

# Binary target: 1 = high need (score >= 14, priority placement threshold), 0 = not high need
df['hpsa_high_need'] = (df['hpsa_score_max'] >= 14).astype(int)

# Verify class balance on designated rows only
designated = df[df['hpsa_designated'] == 1]
counts = designated['hpsa_high_need'].value_counts().sort_index()
print("Class counts:")
print(counts.rename({0: 'not_high_need (0)', 1: 'high_need (1)'}))
print("\nProportions:")
print((counts / counts.sum()).round(3).rename({0: 'not_high_need (0)', 1: 'high_need (1)'}))

In [ ]:
# Feature engineering
def add_features(df):
    df = df.copy()
    df['providers_per_poverty_pop'] = df['pcp_count'] / (df['total_population_poverty'] + 1)
    df['log_total_population']      = np.log1p(df['total_population'])
    df['log_pcp_count']             = np.log1p(df['pcp_count'])
    return df

df = add_features(df)

FEATURES = [
    'uninsured_pct', 'no_checkup_pct',
    'median_household_income', 'poverty_rate_pct',
    'pcp_per_100k',
    'providers_per_poverty_pop', 'log_total_population', 'log_pcp_count'
]
TARGET = 'hpsa_high_need'

print(f"Features ({len(FEATURES)}): {FEATURES}")

In [ ]:
train_df  = df[df['hpsa_designated'] == 1].dropna(subset=FEATURES + [TARGET])
impute_df = df[df['hpsa_designated'] == 0].dropna(subset=FEATURES).copy()

X      = train_df[FEATURES]
y      = train_df[TARGET].astype(int)
groups = train_df['geoid']

# Split by county (geoid) — avoids leaking county-level scores into test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Training tracts : {len(X_train):,}  ({groups.iloc[train_idx].nunique()} counties)")
print(f"Test tracts     : {len(X_test):,}  ({groups.iloc[test_idx].nunique()} counties)")
print(f"Imputation rows : {len(impute_df):,}")

In [ ]:
CLASS_NAMES = ['not_high_need', 'high_need']

def evaluate_model(name, pipeline, config=None):
    pipeline.fit(X_train, y_train)
    preds      = pipeline.predict(X_test)
    pred_proba = pipeline.predict_proba(X_test)[:, 1]
    metrics = {
        "accuracy":    accuracy_score(y_test, preds),
        "weighted_f1": f1_score(y_test, preds, average='weighted'),
        "roc_auc":     roc_auc_score(y_test, pred_proba),
    }
    print(f"\n{'='*45}\n{name}")
    print(classification_report(y_test, preds, target_names=CLASS_NAMES))
    print(f"ROC-AUC: {metrics['roc_auc']:.3f}")
    if WANDB_ENABLED:
        run = wandb.init(
            project="hpsa-classification",
            name=f"{name}_binary",
            config=config or {},
            reinit=True
        )
        wandb.log(metrics)
        run.finish()
    return pipeline, metrics

In [ ]:
trained_models = {}
all_metrics    = {}

if RUN_MODELS['logistic_regression']:
    trained_models['logistic_regression'], all_metrics['logistic_regression'] = evaluate_model(
        'logistic_regression',
        Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=42))]),
        config={'model_type': 'LogisticRegression', 'max_iter': 1000}
    )

if RUN_MODELS['decision_tree']:
    trained_models['decision_tree'], all_metrics['decision_tree'] = evaluate_model(
        'decision_tree',
        Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeClassifier(random_state=42))]),
        config={'model_type': 'DecisionTreeClassifier'}
    )

if RUN_MODELS['random_forest']:
    trained_models['random_forest'], all_metrics['random_forest'] = evaluate_model(
        'random_forest',
        Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))]),
        config={'model_type': 'RandomForestClassifier', 'n_estimators': 100}
    )

if RUN_MODELS['gradient_boosting']:
    trained_models['gradient_boosting'], all_metrics['gradient_boosting'] = evaluate_model(
        'gradient_boosting',
        Pipeline([('scaler', StandardScaler()), ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))]),
        config={'model_type': 'GradientBoostingClassifier', 'n_estimators': 100}
    )

In [ ]:
# Metrics summary table
pd.DataFrame(all_metrics, index=['accuracy', 'weighted_f1', 'roc_auc']).T.round(3).sort_values('roc_auc', ascending=False)

In [ ]:
# Feature importances for tree-based models
for name in ['random_forest', 'gradient_boosting']:
    if name not in trained_models:
        continue
    estimator = trained_models[name].named_steps['model']
    importances = pd.Series(estimator.feature_importances_, index=FEATURES)
    print(f"\n{name} — feature importances:")
    print(importances.sort_values(ascending=False).round(4).to_string())

In [ ]:
# Impute hpsa_high_need for hpsa_designated == 0
X_impute = impute_df[FEATURES]

for name, pipeline in trained_models.items():
    impute_df[f'hpsa_high_need_pred_{name}'] = pipeline.predict(X_impute)

pred_cols = [c for c in impute_df.columns if c.startswith('hpsa_high_need_pred')]
print(impute_df[['geoid_tract', 'geoid'] + pred_cols].head(10))
print(f"\nNull predictions: {impute_df[pred_cols].isnull().sum().sum()}")

In [ ]:
from itertools import product

# --- Gradient Boosting v2: Hyperparameter Tuning ---
param_grid = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
}

gb_v2_results = []

for n_est, depth, lr in product(param_grid['n_estimators'],
                                 param_grid['max_depth'],
                                 param_grid['learning_rate']):
    config = {
        'model_type':    'GradientBoostingClassifier_v2',
        'n_estimators':  n_est,
        'max_depth':     depth,
        'learning_rate': lr,
    }
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  GradientBoostingClassifier(
            n_estimators=n_est,
            max_depth=depth,
            learning_rate=lr,
            random_state=42
        ))
    ])
    pipeline.fit(X_train, y_train)
    preds      = pipeline.predict(X_test)
    pred_proba = pipeline.predict_proba(X_test)[:, 1]
    metrics = {
        'accuracy':    accuracy_score(y_test, preds),
        'weighted_f1': f1_score(y_test, preds, average='weighted'),
        'roc_auc':     roc_auc_score(y_test, pred_proba),
    }
    if WANDB_ENABLED:
        run = wandb.init(
            project="hpsa-classification",
            name="gradient_boosting_v2_binary",
            config=config,
            reinit=True
        )
        wandb.log(metrics)
        run.finish()

    gb_v2_results.append({**config, **metrics, 'pipeline': pipeline})
    print(f"n_est={n_est:3d}  depth={depth}  lr={lr:.2f}  "
          f"acc={metrics['accuracy']:.3f}  f1={metrics['weighted_f1']:.3f}  auc={metrics['roc_auc']:.3f}")

# Pick best by ROC-AUC
best_gb_v2 = max(gb_v2_results, key=lambda x: x['roc_auc'])
print(f"\nBest GB v2 — n_est={best_gb_v2['n_estimators']}  "
      f"depth={best_gb_v2['max_depth']}  lr={best_gb_v2['learning_rate']}  "
      f"AUC={best_gb_v2['roc_auc']:.3f}")

trained_models['gradient_boosting_v2'] = best_gb_v2['pipeline']

In [ ]:
# --- Logistic Regression v2: Hyperparameter Tuning ---
lr_param_grid = {
    'C':       [0.001, 0.01, 0.1, 1.0, 10, 100],
    'penalty': ['l1', 'l2'],
}

lr_v2_results = []

for C, penalty in product(lr_param_grid['C'], lr_param_grid['penalty']):
    config = {
        'model_type': 'LogisticRegression_v2',
        'C':          C,
        'penalty':    penalty,
        'solver':     'liblinear',
    }
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(
            C=C,
            penalty=penalty,
            solver='liblinear',
            max_iter=1000,
            random_state=42
        ))
    ])
    pipeline.fit(X_train, y_train)
    preds      = pipeline.predict(X_test)
    pred_proba = pipeline.predict_proba(X_test)[:, 1]
    metrics = {
        'accuracy':    accuracy_score(y_test, preds),
        'weighted_f1': f1_score(y_test, preds, average='weighted'),
        'roc_auc':     roc_auc_score(y_test, pred_proba),
    }
    if WANDB_ENABLED:
        run = wandb.init(
            project="hpsa-classification",
            name="logistic_regression_v2_binary",
            config=config,
            reinit=True
        )
        wandb.log(metrics)
        run.finish()

    lr_v2_results.append({**config, **metrics, 'pipeline': pipeline})
    print(f"C={C:<8}  penalty={penalty:<2}  "
          f"acc={metrics['accuracy']:.3f}  f1={metrics['weighted_f1']:.3f}  auc={metrics['roc_auc']:.3f}")

# Pick best by ROC-AUC
best_lr_v2 = max(lr_v2_results, key=lambda x: x['roc_auc'])
print(f"\nBest LR v2 — C={best_lr_v2['C']}  penalty={best_lr_v2['penalty']}  "
      f"AUC={best_lr_v2['roc_auc']:.3f}")

trained_models['logistic_regression_v2'] = best_lr_v2['pipeline']

In [ ]:
# --- 4-Category Final Prediction ---
# For HPSA-designated counties: use actual hpsa_high_need
# For non-designated counties:  use tuned model predictions
#
# Categories:
#   0 = hpsa_designated,     low need  (designated=1, high_need=0)
#   1 = hpsa_designated,     high need (designated=1, high_need=1)
#   2 = non-hpsa_designated, low need  (designated=0, predicted=0)
#   3 = non-hpsa_designated, high need (designated=0, predicted=1)

CATEGORY_LABELS = {
    0: 'hpsa_designated_low_need',
    1: 'hpsa_designated_high_need',
    2: 'non_hpsa_low_need',
    3: 'non_hpsa_high_need',
}

def make_4cat(df, pred_col):
    def assign(row):
        if row['hpsa_designated'] == 1:
            return 1 if row['hpsa_high_need'] == 1 else 0
        else:
            return 3 if row[pred_col] == 1 else 2
    return df.apply(assign, axis=1)

df_out = df[['geoid_tract', 'geoid', 'hpsa_designated', 'hpsa_high_need']].copy()

non_designated_mask = df_out['hpsa_designated'] == 0
X_full_impute = df.loc[non_designated_mask, FEATURES]

for model_name in ['gradient_boosting_v2', 'logistic_regression_v2']:
    pred_col = f'predicted_high_need_{model_name}'
    df_out.loc[non_designated_mask, pred_col] = trained_models[model_name].predict(X_full_impute)
    cat_col   = f'hpsa_need_category_4_{model_name}'
    label_col = f'hpsa_need_category_label_{model_name}'
    df_out[cat_col]   = make_4cat(df_out, pred_col)
    df_out[label_col] = df_out[cat_col].map(CATEGORY_LABELS)

    print(f"\n=== {model_name} ===")
    counts = df_out[label_col].value_counts()
    print(counts.to_string())
    print((counts / counts.sum()).round(3).to_string())

df_out[['geoid_tract', 'geoid',
        'hpsa_need_category_label_gradient_boosting_v2',
        'hpsa_need_category_label_logistic_regression_v2']].head(10)